# SIR model of an emerging infection
## Basic model
### Model construction
In this notebook, we will use the `summer` interface 
to demonstrate the construction of a simple model that loosely
captures some of the features of an outbreak of a novel infection.

The model consists of:
- Three compartments, named `susceptible`, `infectious` and `recovered`
- A starting population of seven million people, with one of them infectious
- An evaluation timespan from time zero to 100 days
- Inter-compartmental flows for the infection and recovery processes

Susceptible, infectious, recovered is a very commonly used model structure,
although many variations on this exist,
some of which will be explored in later notebooks.

In [ ]:
%pip install summerepi2==1.3.6

The next cell imports some standard libraries
You don't need to worry about what these are,
although if you have a background in data science
some of them will probably be familiar.
There are some comments to explain what these are
in case you are interested to learn more.

In [ ]:
import pandas as pd  # Pandas for data wrangling
import plotly.express as px  # Plotly for interactive visualisation
import plotly.graph_objects as go
from plotly.subplots import make_subplots
pd.options.plotting.backend = "plotly"

from summer2 import CompartmentalModel  # Our "summer" platform for building epidemic models
from summer2.parameters import Parameter, DerivedOutput
from summer2.functions.time import get_linear_interpolation_function as linear_interp

### Building the model with summer
First, let's create a function that gives us a basic SIR model with its
standard compartments, starting populations and inter-compartmental flows implemented.
![](../images/sir_structure.svg)

Our `summer` library can be used to construct a model with any structure you like,
but we suggest you leave this code unchanged if you're more interested
in the epidemiology.
You can adjust the starting population and the time period
that the simulation will run over at the start of the cell.

In [ ]:
total_population = 7e6
infectious_seed = 1.0
run_period = [0.0, 50.0]
model_comps = ["susceptible", "infectious", "recovered"]
sir_model = CompartmentalModel(times=run_period, compartments=model_comps, infectious_compartments=["infectious"])
start_pop = {"susceptible": total_population - infectious_seed, "infectious": infectious_seed}
sir_model.set_initial_population(start_pop)
sir_model.add_infection_frequency_flow(name="infection", contact_rate=Parameter("contact_rate"), source="susceptible", dest="infectious")
sir_model.add_transition_flow(name="recovery", fractional_rate=Parameter("recovery_rate"), source="infectious", dest="recovered")

### Running the model
Now we have a model set up that we're ready to run.
The model is expecting us to give it some parameters,
because this is what we promised when we said that the
rate of infection and recovery were set with a `Parameter` object.

In [ ]:
parameters = {
    "contact_rate": 1.5,
    "recovery_rate": 0.2,
}
sir_model.run(parameters)
sir_model.get_outputs_df().plot(labels={"index": "time", "value": "number of people"})

### Interpretation
These parameters may not be very intuitive,
because they are rates per unit time.
So for example, a recovery rate of 0.2 per day
implies that if you have a group of infectious people
then approximately 20% of them will recover each day.
We can also say that the average time infectious is five days.

The contact rate in this model is the number of people who 
will be infected by an infectious person each day of their infection.
Because we have already calculated that they will be infectious
for five days, we can calculate the basic reproduction number $R_{0}$
to be 7.5.

That is these simple calculations are:

In [ ]:
infectious_duration = 1.0 / parameters["recovery_rate"]
print(f"The infectious duration is {infectious_duration} days")
basic_reproduction_number = parameters["contact_rate"] * infectious_duration
print(f"The basic reproduction number is {basic_reproduction_number}")

### Limitations
This is a really simple model that doesn't capture many features of
a real infectious disease outbreak.
In particular, there is no random element to it (i.e. it is deterministic)
and the number of people transitioning between the compartments
is calculated by applying the transition rates directly to the calculated compartment sizes.
This means that the calcuated compartment sizes will not be whole numbers.

Nevertheless, it does capture many of the features that we might expect
from a real epidemic of a new infectious disease spreading through
a susceptible population.
Can you list some of these?

## Adding an intervention
### What can we do with a model like this?
So what's the purpose of this model?
Clearly it has several limitations, 
but does also capture quite a few aspects of a real-world epidemic,
including its general shape.
Also, we have now built a "mechanistic" simulation;
that is, we have captured the underlying mechanism of transmission.
This allows us to understand what's happening under the surface
and look at changes to the system that might result from 
public health interventions.

### Incorporating face masks
So next, let's see what happens if we apply a public health intervention
at a point in time.
This gets slightly more complicated in the code,
just because we need to have a process in the model that scales over time.
We'll do this with a simple linear interpolation function,
and parameterise the efficacy and coverage of the intervention.
Let's consider that some proportion of the entire population
starts to wear face masks between day 10 and day 12 of the simulation.

In [ ]:
face_mask_func = linear_interp([10.0, 12.0], [0.0, Parameter("face_mask_coverage")])
contact_rate = (1.0 - face_mask_func * Parameter("face_mask_efficacy")) * Parameter("contact_rate")

total_population = 7e6
infectious_seed = 1.0
run_period = [0.0, 50.0]
model_comps = ["susceptible", "infectious", "recovered"]
sir_model = CompartmentalModel(times=run_period, compartments=model_comps, infectious_compartments=["infectious"])
start_pop = {"susceptible": total_population - infectious_seed, "infectious": infectious_seed}
sir_model.set_initial_population(start_pop)
sir_model.add_infection_frequency_flow(name="infection", contact_rate=contact_rate, source="susceptible", dest="infectious")
sir_model.add_transition_flow(name="recovery", fractional_rate=Parameter("recovery_rate"), source="infectious", dest="recovered")

Now let's run the extended model with the face mask intervention included.
As for the previous model,
you can explore changes to the parameters of this model
to understand the effect of the intervention on the epidemic.
Of course, all of these quantities are quite arbitrary at this stage,
but feel free to replace these numbers with ones that are more
reflective of a particular infectious disease and intervention.

In [ ]:
parameters = {
    "contact_rate": 1.5,
    "recovery_rate": 0.2,
    "face_mask_coverage": 0.8,
    "face_mask_efficacy": 0.5,
}
sir_model.run(parameters)
sir_model.get_outputs_df().plot(labels={"index": "time", "value": "number of people"})